In [1]:
# Importación de librerías
import mne
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
from scipy.stats import shapiro, ttest_rel, wilcoxon
from IPython.display import display
import antropy as ant
from scipy.signal import coherence
from mne.decoding import CSP

In [2]:
# Verificación de carpetas completas (sujetos completos)

# Carpeta de datos
ruta = 'ds004362'

# Detectar carpetas de sujetos
sujetos = [s for s in os.listdir(ruta) if s.startswith('sub-')]

print(sujetos)

['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012', 'sub-013', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018', 'sub-019', 'sub-020', 'sub-021', 'sub-022', 'sub-023', 'sub-024', 'sub-025', 'sub-026', 'sub-027', 'sub-028', 'sub-029', 'sub-030', 'sub-031', 'sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036', 'sub-037', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043', 'sub-044', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050', 'sub-051', 'sub-052', 'sub-053', 'sub-054', 'sub-055', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065', 'sub-066', 'sub-067', 'sub-068', 'sub-069', 'sub-070', 'sub-071', 'sub-072', 'sub-073', 'sub-074', 'sub-075', 'sub-076', 'sub-077', 'sub-078', 'sub-079', 'sub-080', 'sub-081', 'sub-082', 'sub-083', 'sub-084', 'sub-085', 'sub-086', 'sub-087', 'sub-088', 'sub-089', 'sub-090', 'sub-091'

In [3]:
# Función filtrado de señales

def filtrar(senal):
    s_filt = senal.copy().filter(8., 30., fir_window='hamming', verbose=False)

    return s_filt

In [12]:
# Función para calcular PSD por época

def calcular_psd(psd_epoca, freqs, canales, bandas):
    """
    Toma el espectro de potencia de una época y extrae el promedio 
    de energía para las bandas especificadas (mu y beta).
    Retorna los datos en una lista plana de números.
    """
    lista_psd = []
    
    for banda_nombre, (fmin, fmax) in bandas.items():
        # Encontrar los índices de las frecuencias de la banda
        idx_freq = (freqs >= fmin) & (freqs <= fmax)
        # Calcular la potencia promedio en esa banda para todos los canales
        potencia_banda = psd_epoca[:, idx_freq].mean(axis=1)
        
        # Guardar únicamente los valores numéricos en la lista
        for i in range(len(canales)):
            lista_psd.append(potencia_banda[i])
            
    return lista_psd

In [13]:
# Entropía aproximada

def calcular_entropia(datos_epoca):
    """
    Calcula la Entropía Aproximada para cada canal en una sola época.
    datos_epoca: array de numpy con forma (n_canales, n_muestras)
    Retorna: Lista con los valores de ApEn [ApEn_C3, ApEn_C4, ApEn_Cz]
    """

    entrop_aprox = []
    # Iterar sobre cada canal
    for i in range(datos_epoca.shape[0]):
        senal = datos_epoca[i, :]
        entropia = ant.app_entropy(senal, order=2)
        entrop_aprox.append(entropia)
    return entrop_aprox

In [6]:
# Coherencia

def calcular_coherencia_banda(datos_epoca, sfreq, fmin, fmax):
    """
    Calcula el promedio de la coherencia en la banda Mu (8-13 Hz) entre C3 y C4.
    datos_epoca: array de numpy (n_canales, n_muestras). Asumimos 0=C3, 1=C4.
    sfreq: Frecuencia de muestreo.
    Retorna: Un solo valor flotante (Coherencia promedio)
    """

    # Asumimos que C3 es índice 0 y C4 es índice 1
    senal_c3 = datos_epoca[0, :]
    senal_c4 = datos_epoca[1, :]
    
    f, Cxy = coherence(senal_c3, senal_c4, fs=sfreq, nperseg=int(sfreq*2))
    idx_banda = np.where((f >= fmin) & (f <= fmax))[0]
    
    return np.mean(Cxy[idx_banda])

In [7]:
# CSP

def extraer_caracteristicas_csp(epochs_totales, epochs_izq, epochs_der, n_componentes=2):
    """
    Aplica CSP a las épocas de mano izquierda y derecha para extraer sus características (log-varianza).
    Retorna: Matriz de características X_csp y el vector de etiquetas Y.
    """

    # 1. Unir datos para ENTRENAR el CSP (solo Izquierda y Derecha)
    datos_izq = epochs_izq.get_data(copy=True)
    datos_der = epochs_der.get_data(copy=True)
    X_train = np.concatenate([datos_izq, datos_der])
    Y_train = np.concatenate([np.zeros(len(datos_izq)), np.ones(len(datos_der))])
    
    # 2. Entrenar CSP (fit)
    csp = CSP(n_components=n_componentes, reg=None, log=True, norm_trace=False)
    csp.fit(X_train, Y_train)
    
    # 3. Extraer características para TODAS las épocas (transform), incluido el reposo
    X_all = epochs_totales.get_data(copy=True)
    X_caracteristicas = csp.transform(X_all)
    
    return X_caracteristicas

In [18]:
# Función principal: Procesamiento de señales EEG

def procesamiento_eeg(ruta, inicio, fin, archivo_salida="resultados_caracteristicas.csv"):
    
    sujetos = [s for s in os.listdir(ruta) if s.startswith('sub-')]
    sujetos_rango = sujetos[inicio:fin]
    
    if os.path.exists(archivo_salida):
        df_total = pd.read_csv(archivo_salida)
        sujetos_procesados = set(df_total["sujeto"].unique())
        print(f"Archivo existente cargado ({len(sujetos_procesados)} sujetos ya procesados)")
    else:
        df_total = pd.DataFrame()
        sujetos_procesados = set()
    
    for sujeto in sujetos_rango:
        
        if sujeto in sujetos_procesados:
            print(f"{sujeto} ya procesado, se omite")
            continue
            
        print(f"\nProcesando {sujeto}...")
        
        ruta_sujeto = os.path.join(ruta, sujeto, "eeg")
        archivos = [f for f in os.listdir(ruta_sujeto) if f.endswith('.set')]
        runs_imag = ['run-4', 'run-8', 'run-12']
        raws_imag = []

        for archivo in archivos:
            ruta_archivo = os.path.join(ruta_sujeto, archivo)
            if any(run in archivo for run in runs_imag):
                raw = mne.io.read_raw_eeglab(ruta_archivo, preload=True, verbose=False)
                raws_imag.append(raw)

        raw_imag = mne.concatenate_raws(raws_imag)

        # Selección de canales
        canales = ['C3', 'C4']
        raw_imag.pick_channels(canales)
        
        # Filtrado base
        imag_filtrado = filtrar(raw_imag)
        
        events_i, event_id_i = mne.events_from_annotations(imag_filtrado, verbose=False)
        epochs_imag = mne.Epochs(imag_filtrado, events_i, event_id=event_id_i, tmin=0, tmax=4, 
                                 baseline=None, preload=True, verbose=False)
        
        sfreq = raw_imag.info['sfreq']
        
        bandas = {
            "mu": (8, 12),
            "beta": (13, 30)
        }
        
        # Copias filtradas para las otras métricas
        epochs_mu = epochs_imag.copy().filter(bandas["mu"][0], bandas["mu"][1], verbose=False)
        epochs_beta = epochs_imag.copy().filter(bandas["beta"][0], bandas["beta"][1], verbose=False)
        
        csp_mu = extraer_caracteristicas_csp(epochs_mu, epochs_mu['TASK2T1'], epochs_mu['TASK2T2'], n_componentes=2)
        csp_beta = extraer_caracteristicas_csp(epochs_beta, epochs_beta['TASK2T1'], epochs_beta['TASK2T2'], n_componentes=2)
        
        # CÁLCULO VECTORIZADO DE PSD GLOBAL
        psd_global, freqs = mne.time_frequency.psd_array_welch(
            epochs_imag.get_data(copy=True), sfreq=sfreq, fmin=1, fmax=40, verbose=False
        )

        datos_mu = epochs_mu.get_data(copy=True)
        datos_beta = epochs_beta.get_data(copy=True)
        id_a_condicion = {v: k for k, v in event_id_i.items()}

        filas_sujeto = []

        for epoca_idx in range(len(epochs_imag)):
            
            id_evento = epochs_imag.events[epoca_idx, 2]
            cond_str = id_a_condicion.get(id_evento, "")
            
            # ETIQUETAS NUMÉRICAS DIRECTAS
            if 'TASK2T0' in cond_str: tarea = 0
            elif 'TASK2T1' in cond_str: tarea = 1
            elif 'TASK2T2' in cond_str: tarea = 2
            else: continue 
            
            fila = {
                "sujeto": sujeto,
                "tarea": tarea,
                "epoca": epoca_idx
            }
            
            # =========================================================
            # A. EXTRAER PSD (Ahora proviene de una lista)
            # =========================================================
            lista_valores_psd = calcular_psd(psd_global[epoca_idx], freqs, canales, bandas)
            
            # Asignamos los valores manualmente desde la lista a la fila.
            # El orden de la lista es: [mu_C3, mu_C4, mu_Cz, beta_C3, beta_C4, beta_Cz]
            fila["PSD_mu_C3"] = lista_valores_psd[0]
            fila["PSD_mu_C4"] = lista_valores_psd[1]
            
            fila["PSD_beta_C3"] = lista_valores_psd[2]
            fila["PSD_beta_C4"] = lista_valores_psd[3]
            
            # =========================================================
            # B. Entropía Aproximada
            # =========================================================
            apen_mu = calcular_entropia(datos_mu[epoca_idx])
            apen_beta = calcular_entropia(datos_beta[epoca_idx])
            for i, ch in enumerate(canales):
                fila[f"ApEn_mu_{ch}"] = apen_mu[i]
                fila[f"ApEn_beta_{ch}"] = apen_beta[i]
                
            # =========================================================
            # C. Coherencia (C3-C4)
            # =========================================================
            fila["Coh_mu_C3_C4"] = calcular_coherencia_banda(datos_mu[epoca_idx], sfreq, bandas["mu"][0], bandas["mu"][1])
            fila["Coh_beta_C3_C4"] = calcular_coherencia_banda(datos_beta[epoca_idx], sfreq, bandas["beta"][0], bandas["beta"][1])
            
            # =========================================================
            # D. Características CSP
            # =========================================================
            fila["CSP_mu_comp1"] = csp_mu[epoca_idx, 0]
            fila["CSP_mu_comp2"] = csp_mu[epoca_idx, 1]
            fila["CSP_beta_comp1"] = csp_beta[epoca_idx, 0]
            fila["CSP_beta_comp2"] = csp_beta[epoca_idx, 1]
            
            filas_sujeto.append(fila)
            
        df_sujeto = pd.DataFrame(filas_sujeto)
        df_sujeto.to_csv(archivo_salida, mode='a', header=not os.path.exists(archivo_salida), index=False)
        
        print(f"--> {sujeto} listo.")

    print("Sujetos procesados - Proceso finalizado")

In [19]:
procesamiento_eeg(ruta, inicio=0, fin=109)


Procesando sub-001...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Computing rank from data with rank=None
    Using tolerance 1.3e-06 (2.2e-16 eps * 2 dim * 2.9e+09  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
Reducing data rank from 2 -> 2
Estimating class=0.0 covariance using EMPIRICAL
Done.
Estimating class=1.0 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.4e-06 (2.2e-16 eps * 2 dim * 3.1e+09  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
Reducing data rank from 2 -> 2
Estimating class=0.0 covariance using EMPIRICAL
Done.
Estimating class=1.0 covariance using EMPIRICAL
Done.
--> sub-001 listo.

Procesando sub-002...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Computing rank from data with rank=None
    Using tolerance 7.2e-07 (

In [20]:
df = pd.read_csv("resultados_caracteristicas.csv")
display(df.head(10))

,sujeto,tarea,epoca,PSD_mu_C3,PSD_mu_C4,PSD_beta_C3,PSD_beta_C4,ApEn_mu_C3,ApEn_beta_C3,ApEn_mu_C4,ApEn_beta_C4,Coh_mu_C3_C4,Coh_beta_C3_C4,CSP_mu_comp1,CSP_mu_comp2,CSP_beta_comp1,CSP_beta_comp2
0,sub-001,0,0,1.164672e-11,1.417487e-11,7.682990e-12,4.981189e-12,0.570400,0.886904,0.599652,0.976759,0.669231,0.609161,-1.094025,-0.793854,-1.004695,-1.088702
1,sub-001,2,1,1.687640e-11,1.060752e-11,8.244530e-12,8.524374e-12,0.578993,0.890194,0.590078,0.864570,0.646239,0.677454,-1.520772,-1.221679,-1.128982,-0.859706
2,sub-001,0,2,3.528920e-11,2.464886e-11,7.797381e-12,4.905408e-12,0.571612,0.976583,0.577775,1.051539,0.711312,0.617336,-0.589068,-0.692220,-0.671218,-1.075737
3,sub-001,1,3,2.653582e-11,2.607450e-11,1.022708e-11,8.645903e-12,0.576481,0.889757,0.586594,0.881570,0.727851,0.615471,-0.615036,-0.879434,-0.972503,-0.799456
4,sub-001,0,4,3.834172e-11,3.649078e-11,7.962012e-12,5.495447e-12,0.597163,0.911826,0.595641,0.852669,0.866652,0.606289,-0.964911,-0.618937,-0.887249,-0.786711
5,sub-001,2,5,1.980877e-11,2.005997e-11,7.099971e-12,4.857837e-12,0.567511,0.916004,0.560001,0.971846,0.735539,0.523273,-1.075399,-1.001390,-0.922628,-1.214172
6,sub-001,0,6,3.397905e-11,2.347461e-11,1.185468e-11,7.843826e-12,0.560620,0.921381,0.605512,0.959796,0.580562,0.655901,-0.420203,-0.357011,-0.524810,-0.606461
7,sub-001,1,7,2.004713e-11,1.154575e-11,7.718975e-12,4.553753e-12,0.568765,0.960377,0.544803,0.948502,0.740117,0.578058,-0.677710,-1.228096,-0.769802,-1.065826
8,sub-001,0,8,2.614896e-11,2.310862e-11,8.266642e-12,6.031115e-12,0.583924,0.872035,0.571143,0.921557,0.732227,0.666664,-0.593083,-0.858830,-0.619524,-0.905999
9,sub-001,1,9,8.367194e-11,2.446277e-11,1.082363e-11,8.390492e-12,0.551311,0.930532,0.563831,0.916404,0.754648,0.626342,-0.030796,-0.795509,-0.695216,-0.728047
